# Software Documentation Assistant: Retrieval & Evaluation Benchmark
## Week 3 | Retrieval-Augmented Generation Applications (Part C)

This notebook executes the 15-question benchmark evaluation, measuring:
1. Version Sensitivity (FastAPI 0.115 vs 0.90, Pydantic V2 vs V1)
2. API Explanation & Code Snippet Retrieval
3. Citation Correctness & Grounding
4. Unanswerable Question Abstention (Hallucination Prevention)
5. Latency and Quantitative IR Metrics (MRR, Hit Rate@4, Precision@4)

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.rag_pipeline import RAGPipeline
from src.evaluator import BenchmarkEvaluator, BENCHMARK_15_QUESTIONS
import pandas as pd
import matplotlib.pyplot as plt
import json

pipeline = RAGPipeline()
pipeline.build_or_load_index(force_reindex=False)
evaluator = BenchmarkEvaluator(pipeline)

[INFO] Vector store loaded from C:\Users\ROSHAN ALEXANDER\Documents\Projects\Software-Documentation-Assistant\results\index_cache (75 chunks).
[INFO] Loaded existing index (75 chunks) in 0.01s.


### 1. Interactive Single Query Demonstration
Let's test an answerable migration query and verify citations and code.

In [2]:
query = "How do I migrate from BaseModel.dict() to Pydantic v2 syntax?"
response = pipeline.query(query)

print(f"Query: {response['query']}")
print(f"Abstained: {response['is_abstained']}")
print(f"Confidence Score: {response['confidence_score']}")
print(f"Citations: {response['citations']}")
print(f"Latency: {response['latency']['total_ms']}ms\n")
print("--- Generated Answer ---")
print(response['answer'])

[INFO] Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Embedding model loaded successfully.


Query: How do I migrate from BaseModel.dict() to Pydantic v2 syntax?
Abstained: False
Confidence Score: 0.8243
Citations: ["[Doc: DOC-PYD-001 | Section: 'Key Syntax Changes: V1 to V2' | Ver: v2.0]"]
Latency: 22146.64ms

--- Generated Answer ---
### Technical Overview (Pydantic v2.0)

According to the official Pydantic documentation (Pydantic V2 Architecture, Migration Guide, and New Syntax), in section **'Key Syntax Changes: V1 to V2'** (Target Version: `v2.0`):

| Feature / Intent | Pydantic V1 (Deprecated) | Pydantic V2 (Current Standard) | | :--- | :--- | :--- | | Export model to Python dict | `model.dict()` | `model.model_dump()` | [Doc: DOC-PYD-001 | Section: 'Key Syntax Changes: V1 to V2' | Ver: v2.0]


### Version Comparison & Deprecation Notice

Regarding version `v2.0`: from typing import Optional from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator [Doc: DOC-PYD-001 | Section: 'Modern Pydantic V2 Code Example' | Ver: v2.0]



### 2. Abstention Test on Out-of-Scope Query
Testing whether the system safely refuses to hallucinate on fictitious features.

In [3]:
out_of_scope_query = "What is the @app.quantum_accelerator() decorator in FastAPI for quantum computing speedups?"
refusal_response = pipeline.query(out_of_scope_query)

print(f"Query: {refusal_response['query']}")
print(f"Abstained: {refusal_response['is_abstained']}")
print(f"Confidence Score: {refusal_response['confidence_score']}")
print("--- Abstention Response ---")
print(refusal_response['answer'])

Query: What is the @app.quantum_accelerator() decorator in FastAPI for quantum computing speedups?
Abstained: True
Confidence Score: 0.6869
--- Abstention Response ---
[ABSTENTION] Based on the indexed technical documentation, there is insufficient evidence to answer the question: 'What is the @app.quantum_accelerator() decorator in FastAPI for quantum computing speedups?'. The knowledge base does not contain matching API explanations or code examples for this query (Top similarity score: 0.687 below threshold 0.38).


### 3. Run Complete 15-Question Benchmark Suite
Calculates quantitative metrics across all categories.

In [4]:
bench_results = evaluator.run_benchmark(BENCHMARK_15_QUESTIONS)
summary = bench_results['summary']

print("\n================ BENCHMARK SUMMARY ================")
print(f"Total Questions: {summary['total_questions']}")
print(f"Hit Rate@4: {summary['hit_rate_at_k']*100:.1f}%")
print(f"MRR (Mean Reciprocal Rank): {summary['mrr']:.4f}")
print(f"Citation Correctness: {summary['citation_correctness_pct']}%")
print(f"Abstention Accuracy: {summary['abstention_accuracy_pct']}%")
print(f"Confusion Matrix: {summary['confusion_matrix']}")
print(f"Median Latency: {summary['latency_stats']['median_ms']}ms")


Starting Benchmark Evaluation on 15 Standard Questions

[✓] Q01 (Version Sensiti): Score=0.824 | Abstained=False | 8.8ms
[✓] Q02 (Version Sensiti): Score=1.000 | Abstained=False | 9.3ms
[✓] Q03 (Version Sensiti): Score=0.913 | Abstained=False | 10.2ms
[✓] Q04 (Version Sensiti): Score=0.873 | Abstained=False | 8.8ms
[✓] Q05 (Version Sensiti): Score=0.823 | Abstained=False | 8.3ms
[✓] Q06 (API & Code Exam): Score=0.805 | Abstained=False | 8.8ms
[✓] Q07 (API & Code Exam): Score=0.834 | Abstained=False | 8.8ms
[✓] Q08 (API & Code Exam): Score=0.804 | Abstained=False | 9.0ms
[✓] Q09 (API & Code Exam): Score=0.811 | Abstained=False | 9.1ms
[✓] Q10 (API & Code Exam): Score=0.808 | Abstained=False | 9.7ms
[✓] Q11 (Architecture & ): Score=0.824 | Abstained=False | 23.8ms
[✓] Q12 (Architecture & ): Score=0.872 | Abstained=False | 19.5ms
[✓] Q13 (Abstention & Ne): Score=0.591 | Abstained=True | 15.4ms
[✓] Q14 (Abstention & Ne): Score=0.687 | Abstained=True | 16.2ms
[✓] Q15 (Abstention & Ne): Sco

[INFO] Saved evaluation charts to C:\Users\ROSHAN ALEXANDER\Documents\Projects\Software-Documentation-Assistant\results\evaluation_metrics.png and confusion_matrix.png

================ BENCHMARK SUMMARY ================
Total Questions: 15
Hit Rate@4: 100.0%
MRR (Mean Reciprocal Rank): 1.0000
Citation Correctness: 100.0%
Abstention Accuracy: 100.0%
Confusion Matrix: {'TP': 12, 'TN': 3, 'FP': 0, 'FN': 0}
Median Latency: 9.34ms


### 4. Results Breakdown Table

In [5]:
df = pd.DataFrame(bench_results['results'])[['id', 'category', 'question', 'confidence_score', 'is_abstained', 'abstention_status', 'latency_ms']]
df

,id,category,question,confidence_score,is_abstained,abstention_status,latency_ms
0,1,Version Sensitivity,How do I migrate from BaseModel.dict() to Pyda...,0.8243,False,CORRECT_ANSWER,8.79
1,2,Version Sensitivity,What is the recommended modern way to manage s...,1.0000,False,CORRECT_ANSWER,9.34
2,3,Version Sensitivity,How were startup events registered in legacy F...,0.9134,False,CORRECT_ANSWER,10.20
3,4,Version Sensitivity,How do field validators differ between Pydanti...,0.8731,False,CORRECT_ANSWER,8.83
4,5,Version Sensitivity,What replaced the regex parameter in Query and...,0.8231,False,CORRECT_ANSWER,8.33
5,6,API & Code Examples,How do I declare dependency injection using ty...,0.8047,False,CORRECT_ANSWER,8.81
6,7,API & Code Examples,How do I implement OAuth2 password bearer toke...,0.8341,False,CORRECT_ANSWER,8.77
7,8,API & Code Examples,How can I create custom middleware using BaseH...,0.8044,False,CORRECT_ANSWER,9.00
8,9,API & Code Examples,How do asynchronous BackgroundTasks work in Fa...,0.8110,False,CORRECT_ANSWER,9.07
9,10,API & Code Examples,How do I stream generative LLM tokens using St...,0.8079,False,CORRECT_ANSWER,9.65
